# 文本模型 vicuna-7b 测试   —— 有点问题

# 文本模型 vicuna-7b-v1.5 测试

In [10]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("/home/NCUT/teacher/xmy/MRAG/PMI/hugging_cache/vicuna-7b-v1.5")
model = AutoModelForCausalLM.from_pretrained("/home/NCUT/teacher/xmy/MRAG/PMI/hugging_cache/vicuna-7b-v1.5")

Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.62s/it]
/home/NCUT/teacher/xmy/anaconda3/envs/Kedkg/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/home/NCUT/teacher/xmy/anaconda3/envs/Kedkg/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`. This was detected when initializing the generation config instance, which means the corresponding file may hold inco

In [2]:
prompt = "USER:Introduce China briefly please \nASSITANT:"
inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    do_sample=True,
    top_p=0.9,
    temperature=0.7,
    eos_token_id=tokenizer.eos_token_id,
    no_repeat_ngram_size=2,
    early_stopping=True
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/home/NCUT/teacher/xmy/anaconda3/envs/Kedkg/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:615: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


USER:Introduce China briefly please 
ASSITANT: China is a country located in East Asia. It is the world's most populous country, with over 1.4 billion people, and the second-largest economy in the


In [4]:
prompt = "Introduce China briefly please"

In [15]:
def ask_vicuna(user_prompt):
    template = (
        # "A chat between a curious user and an artificial intelligence assistant.\n"
        # "The assistant gives helpful, detailed, and polite answers to the user's questions.\n\n"
        # 上面两个prompt有点太长了，没有加，因为会占用max_new_tokens的字数，直接使用USER ASSISTANT 模板
        f"USER: {user_prompt}\n"
        "ASSISTANT:"
    )
    
    inputs = tokenizer(template, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=2,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [11]:
device_id = 4  # 你想用的 GPU 编号，例如：cuda:2
device = torch.device(f"cuda:{device_id}" if torch.cuda.is_available() else "cpu")


In [16]:
model.to(device)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=0)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      

In [17]:
generated_text = ask_vicuna(prompt)

/home/NCUT/teacher/xmy/anaconda3/envs/Kedkg/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:615: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


In [18]:
# 提取回答中的answer之后的部分
def extract_answer(text: str, keyword: str = "ASSISTANT:") -> str:
    if keyword in text:
        return text.split(keyword, 1)[1].strip()
    else:
        return text.strip()

In [19]:
Answer_real_answer = extract_answer(generated_text)

In [20]:
Answer_real_answer

"China is a country located in East Asia, with a population of over 1.4 billion people. It is the world's most populous country and the second-largest economy, after the United States. China has a long and rich history, dating back thousands of years, and has played a significant role in world history and culture. Today, China continues to be a major global power, investing heavily in technology, infrastructure,"

In [20]:
template = (
        # "A chat between a curious user and an artificial intelligence assistant.\n"
        # "The assistant gives helpful, detailed, and polite answers to the user's questions.\n\n"
        # 上面两个prompt有点太长了，没有加，因为会占用max_new_tokens的字数，直接使用USER ASSISTANT 模板
        "USER:"
        "ASSISTANT:"
    )

In [21]:
template


'USER:ASSISTANT:'